# **HANDS - ON: DISEÑO DE AGENTES CONVERSACIONALES**

Una vez vista la masterclass ***Diseño de Agentes Conversacionales***, se proporciona el siguiente ***colab*** para que **el alumnado pueda practicar** haciendo uso del lenguaje de programación ***python*** y el modelo de ***llama 3.1*** de Meta.


De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/18wkM2LSfY39VZ2GBdcPCY2Qw0EzewJsx?usp=drive_link).

## **LÓGICA DE NEGOCIO Y MODELO**

### **CONFIGURACIÓN DEL ENTORNO**


#### **COLAB SECRETS**

Para no exponer tu API key directamente en el código, Colab ofrece un panel de Secrets (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la API key para guardarla dentro de una variable y usarla dentro del notebook.

In [ ]:
#Leer credenciales desde Colab Secrets

from google.colab import userdata

VERIFY_TOKEN = "mi_token"
WHATSAPP_TOKEN = userdata.get('WHATSAPP_ACCESS_TOKEN')
PHONE_ID = userdata.get('PHONE_NUMBER_ID')

print("Credenciales cargadas correctamente.")


Credenciales cargadas correctamente.


#### **INSTALAR Y ARRANCAR OLLAMA EN COLAB**

Este módulo corre en Colab, así que Ollama no viene preinstalado en la máquina virtual: se instala y se arranca en segundo plano al inicio de cada sesión.

In [ ]:
# Instalar Ollama

!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh


W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 135 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (1,116 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack ...

In [ ]:
# Iniciar el servidor de Ollama en segundo plano

import subprocess
import time

proceso_ollama = subprocess.Popen(["ollama", "serve"])
time.sleep(5)

!pip install ollama --quiet
import ollama

!ollama pull llama3.1:8b
print("Ollama listo.")



Ollama listo.


#### **INSTALAR Y LEVANTAR LLAMA STACK**

In [ ]:
# Instalar dependencias

!pip install "llama-stack[starter]" llama-stack-client --quiet
!pip install "mcp<2.0" --quiet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 29.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.3/375.3 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.8/952.8 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 569.2/569.2 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.7/319.7 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.0/80.0 kB 7.8 MB/s eta 0:00:00
  

In [ ]:
# Instalar Llama Stack (servidor + cliente) y levantarlo con Ollama registrado

import subprocess
import time
import os
import yaml
import llama_stack

os.environ["OLLAMA_URL"] = "http://localhost:11434/v1"

# Generar un config de Llama Stack con el modelo de Ollama ya registrado
config_path = os.path.join(
    os.path.dirname(llama_stack.__file__),
    "distributions",
    "starter",
    "config.yaml"
)

with open(config_path) as f:
    config = yaml.safe_load(f)

config["registered_resources"]["models"].append({
    "model_id": "llama3.1:8b",
    "provider_id": "ollama",
    "provider_model_id": "llama3.1:8b",
    "model_type": "llm",
})

mi_config_path = "/content/mi_config_llama_stack.yaml"

with open(mi_config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

# Levantar el servidor con ese config
proceso_llama_stack = subprocess.Popen(
    ["llama", "stack", "run", mi_config_path, "--port", "8321"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(20)  # tarda mas en levantar que Ollama

print("Proceso vivo:", proceso_llama_stack.poll() is None)

if proceso_llama_stack.poll() is not None:
    print("--- Salida del proceso ---")
    print(proceso_llama_stack.stdout.read())
else:
    from llama_stack_client import LlamaStackClient

    client = LlamaStackClient(base_url="http://localhost:8321")
    print("Llama Stack listo.")

Proceso vivo: True
Llama Stack listo.


### **CONSTRUCCIÓN DE LA LÓGICA DEL ASISTENTE**

In [ ]:
# Configurar el servidor del webhook

from fastapi import FastAPI, Request, BackgroundTasks
import requests
import ollama

app = FastAPI()

mensajes_recibidos = []
usuarios_conocidos = set()


In [ ]:
# Definir función de normalización de número

def normalizar_numero_mx(numero):
    if numero.startswith("521") and len(numero) == 13:
        return "52" + numero[3:]
    return numero


#### **CONSTRUIR LA MEMORIA DE CONVERSACIÓN**

Se define una estructura en memoria que guarda, por cada número de teléfono, el historial reciente de mensajes — sin esto, Llama trataría cada mensaje como si fuera el primero de la conversación. Esta es una implementación manual y simplificada del concepto; la lectura del Tema menciona **Llama Stack** como el framework que estandariza esta pieza (junto con herramientas y seguridad) para no reconstruirla en cada proyecto. **Nota:** el patrón exacto de integración de Llama Stack debe confirmarse contra su documentación vigente antes de grabar o dar la sesión en vivo.

In [ ]:
# Una conversacion de Llama Stack por numero de telefono

conversaciones_por_numero = {}

def obtener_conversacion(numero):
    if numero not in conversaciones_por_numero:
        conversacion = client.conversations.create()
        conversaciones_por_numero[numero] = conversacion.id
    return conversaciones_por_numero[numero]


#### **GENERAR LA RESPUESTA CON HISTORIAL Y UNA REGLA FIJA**

Se define la lógica que arma cada respuesta: si el usuario pide hablar con una persona, se responde con un mensaje fijo de escalamiento (una decisión que conviene fijar por regla, no dejarla al modelo); en cualquier otro caso, se reenvía todo el historial reciente a Llama, para que pueda resolver referencias a mensajes anteriores.

In [ ]:
# Definir la funcion de generacion de respuesta con memoria

FRASES_ESCALAMIENTO = [
    "hablar con alguien",
    "hablar con una persona",
    "agente humano"
]

SYSTEM_PROMPT = (
    "Eres el asistente de agendado de citas de una clinica dental. "
    "Responde de forma breve (2-3 lineas)."
)

def generar_respuesta(numero, texto_usuario):
    texto_low = texto_usuario.lower()

    if any(frase in texto_low for frase in FRASES_ESCALAMIENTO):
        return "Claro, en un momento un miembro del equipo te contacta directamente."

    conversation_id = obtener_conversacion(numero)

    respuesta = client.responses.create(
        model="ollama/llama3.1:8b",
        input=texto_usuario,
        conversation=conversation_id,
        instructions=SYSTEM_PROMPT,
    )

    return respuesta.output_text


## **DESPLIEGUE DEL SERVIDOR**

### **DEFINIR LOS ENDPOINTS DEL WEBHOOK**

Se definen los dos endpoints que WhatsApp necesita: uno para la verificación inicial del webhook, y otro para recibir cada mensaje entrante. La recepción delega el trabajo pesado a `BackgroundTasks`, para responderle a Meta de inmediato sin esperar a que Llama genere la respuesta — evitando así que Meta reintente la entrega y se dupliquen las respuestas.

In [ ]:
# Definir endpoint de verificacion de credenciales

@app.get("/webhook")
async def verificar_webhook(request: Request):
    params = request.query_params

    if params.get("hub.verify_token") == VERIFY_TOKEN:
        return int(params.get("hub.challenge"))

    return {"error": "token invalido"}

In [ ]:
# Definir endpoint de recepcion de mensajes

@app.post("/webhook")
async def recibir_mensaje(request: Request, background_tasks: BackgroundTasks):
    body = await request.json()
    mensajes_recibidos.append(body)

    try:
        mensaje = body["entry"][0]["changes"][0]["value"]["messages"][0]
        numero = normalizar_numero_mx(mensaje["from"])
        texto = mensaje["text"]["body"]

        background_tasks.add_task(
            procesar_y_responder,
            numero,
            texto
        )

    except (KeyError, IndexError):
        pass

    return {"status": "ok"}


In [ ]:
# Definir funcion de envio de respuesta

respuestas_enviadas = []

def enviar_respuesta(numero, texto):
    url = f"https://graph.facebook.com/v25.0/{PHONE_ID}/messages"

    headers = {
        "Authorization": f"Bearer {WHATSAPP_TOKEN}",
        "Content-Type": "application/json"
    }

    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {"body": texto}
    }

    r = requests.post(url, headers=headers, json=payload)

    respuestas_enviadas.append({
        "numero": numero,
        "status": r.status_code,
        "texto": r.text
    })


In [ ]:
# Definir funcion de procesamiento: regla fija de escalamiento o respuesta de Llama

errores_procesamiento = []

def procesar_y_responder(numero, texto):
    with obtener_lock(numero):
        try:
            texto_low = texto.lower()

            if any(frase in texto_low for frase in FRASES_ESCALAMIENTO):
                enviar_respuesta(
                    numero,
                    "Claro, en un momento un miembro del equipo te contacta directamente."
                )
            else:
                respuesta = generar_respuesta(numero, texto)
                enviar_respuesta(numero, respuesta)

        except Exception as e:
            errores_procesamiento.append({
                "numero": numero,
                "texto": texto,
                "error": repr(e)
            })


In [ ]:
# Un lock por numero de telefono, para serializar mensajes del mismo usuario

import threading

locks_por_numero = {}  # { numero: threading.Lock() }

def obtener_lock(numero):
    if numero not in locks_por_numero:
        locks_por_numero[numero] = threading.Lock()
    return locks_por_numero[numero]

### **EXPONER EL SERVIDOR CON NGROK**

Se abre un túnel público con ngrok y se levanta el servidor en un hilo aparte, para que Meta pueda entregarle los mensajes reales al webhook. La URL impresa (+ `/webhook`) es la que se configura como *Callback URL* en Meta for Developers (WhatsApp > Configuración de la API > Configurar webhooks).

In [ ]:
# Instalar e iniciar tunel publico con ngrok

!pip install pyngrok --quiet

from pyngrok import ngrok
import uvicorn
import threading

ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
tunel_publico = ngrok.connect(8000)

print("URL a configurar como Callback URL en el dashboard de Meta:")
print(tunel_publico.public_url + "/webhook")

def correr_servidor():
    uvicorn.run(app, host="0.0.0.0", port=8000)

hilo = threading.Thread(target=correr_servidor, daemon=True)
hilo.start()

URL a configurar como Callback URL en el dashboard de Meta:
https://trustee-hardware-antsy.ngrok-free.dev/webhook


### **PROBAR EL FLUJO COMPLETO**

Se manda un primer mensaje mencionando el servicio dental, y un segundo mensaje que depende del primero — para confirmar que el historial sí se está reenviando.

In [ ]:
# Mandar el primer mensaje, mencionando el servicio

payload_1 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "526182414360", "text": {"body": "Quiero una cita dental"}}
    ]}}]}]
}

r1 = requests.post("http://127.0.0.1:8000/webhook", json=payload_1)
print("Mensaje 1:", r1.status_code, r1.text)

INFO:     127.0.0.1:48396 - "POST /webhook HTTP/1.1" 200 OK
Mensaje 1: 200 {"status":"ok"}


In [ ]:
# Mandar un segundo mensaje que depende del primero, sin repetir el servicio

payload_2 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "526182414360", "text": {"body": "¿Tienen horario el sábado?"}}
    ]}}]}]
}

r2 = requests.post("http://127.0.0.1:8000/webhook", json=payload_2)
print("Mensaje 2:", r2.status_code, r2.text)

# la respuesta demora aprox. de 1 a 2 min. al ejecutar local en colab

INFO:     127.0.0.1:45736 - "POST /webhook HTTP/1.1" 200 OK
Mensaje 2: 200 {"status":"ok"}


In [ ]:
# Verificar respuestas
respuestas_enviadas

[{'numero': '526182414360',
  'status': 200,
  'texto': '{"messaging_product":"whatsapp","contacts":[{"input":"526182414360","wa_id":"5216182414360"}],"messages":[{"id":"wamid.HBgNNTIxNjE4MjQxNDM2MBUCABEYEjFEQjk1ODIyQjkyMUY1OUIxMgA="}]}'},
 {'numero': '526182414360',
  'status': 200,
  'texto': '{"messaging_product":"whatsapp","contacts":[{"input":"526182414360","wa_id":"5216182414360"}],"messages":[{"id":"wamid.HBgNNTIxNjE4MjQxNDM2MBUCABEYEjRGQkM2NDg1NDNDNkY4OEFBQwA="}]}'}]

In [ ]:
# Ver mensajes recibidos
mensajes_recibidos

[{'entry': [{'changes': [{'value': {'messages': [{'from': '526182414360',
         'text': {'body': 'Quiero una cita dental'}}]}}]}]},
 {'entry': [{'changes': [{'value': {'messages': [{'from': '526182414360',
         'text': {'body': '¿Tienen horario el sábado?'}}]}}]}]},
 {'entry': [{'changes': [{'value': {'messages': [{'from': '526182414360',
         'text': {'body': 'Quiero una cita dental'}}]}}]}]},
 {'entry': [{'changes': [{'value': {'messages': [{'from': '526182414360',
         'text': {'body': '¿Tienen horario el sábado?'}}]}}]}]},
 {'entry': [{'changes': [{'value': {'messages': [{'from': '526182414360',
         'text': {'body': 'Quiero una cita dental'}}]}}]}]},
 {'entry': [{'changes': [{'value': {'messages': [{'from': '526182414360',
         'text': {'body': '¿Tienen horario el sábado?'}}]}}]}]}]

In [ ]:
# Verificar almacenamiento en Llama Stack

numero = normalizar_numero_mx("526182414360")  # tu numero real de WhatsApp
conv_id = conversaciones_por_numero[numero]

items = client.conversations.items.list(conversation_id=conv_id)
for item in items.data:
    print(item.role, "->", item.content)


assistant -> [OpenAIResponseMessageOutputContentListOpenAIResponseOutputMessageContentOutputTextOutputOpenAIResponseContentPartRefusalOpenAIResponseOutputMessageContentOutputTextOutput(text='Sí, tenemos citas disponibles los sábados. ¿Cuándo te gustaría venir? ¿Estarías dispondo para la mañana o la tarde?', annotations=[], logprobs=[], type='output_text')]
user -> [OpenAIResponseMessageOutputContentListOpenAIResponseInputMessageContentTextOpenAIResponseInputMessageContentImageOpenAIResponseInputMessageContentFileOpenAIResponseInputMessageContentText(text='¿Tienen horario el sábado?', type='input_text')]
assistant -> [OpenAIResponseMessageOutputContentListOpenAIResponseOutputMessageContentOutputTextOutputOpenAIResponseContentPartRefusalOpenAIResponseOutputMessageContentOutputTextOutput(text='Por supuesto. ¿Quieres hacer una cita para un examen general, odontológico, limpieza de boca o tienes otro problema dental en particular que deseas tratar?', annotations=[], logprobs=[], type='outpu

In [ ]:
# Finalizar proceso de tunelización

ngrok.kill()
print('Proceso de ngrok cerrado.')


Proceso de ngrok cerrado.


## **CHALLENGE: AGENTE DE UNA VETERINARIA**

Una vez visto el ***Hands-On: Diseño de Agentes Conversacionales***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un agente similar, pero para una veterinaria: además de la regla de escalamiento a una persona, se agrega una regla de urgencia médica, y un límite al historial que se conserva en la conversación de Llama Stack.

**IMPORTANTE:** Para su revisión, es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.

### **INSTRUCCIONES:**

**1. Prepara el entorno:** Instala y arranca Ollama en Colab, descarga el modelo Llama 3.1 8B, instala y levanta **Llama Stack** con Ollama registrado, y carga las credenciales de WhatsApp desde **Colab Secrets**.

In [1]:
# Instalar Ollama (requiere zstd, que Colab no trae por defecto)

!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh


W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 140 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (550 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../z

In [2]:
# Iniciar el servidor de Ollama en segundo plano

import subprocess
import time

proceso_ollama = subprocess.Popen(["ollama", "serve"])
time.sleep(5)

!pip install ollama --quiet
import ollama

!ollama pull llama3.1:8b
print("Ollama listo.")



Ollama listo.


In [3]:
# Instalar dependencias

!pip install "llama-stack[starter]" llama-stack-client --quiet
!pip install "mcp<2.0" --quiet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 27.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.3/375.3 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.8/952.8 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 569.2/569.2 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.7/319.7 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.0/80.0 kB 7.3 MB/s eta 0:00:00
  

In [4]:
# Instalar Llama Stack (servidor + cliente) y levantarlo con Ollama registrado

import subprocess
import time
import os
import yaml
import llama_stack

os.environ["OLLAMA_URL"] = "http://localhost:11434/v1"

config_path = os.path.join(
    os.path.dirname(llama_stack.__file__),
    "distributions",
    "starter",
    "config.yaml"
)

with open(config_path) as f:
    config = yaml.safe_load(f)

config["registered_resources"]["models"].append({
    "model_id": "llama3.1:8b",
    "provider_id": "ollama",
    "provider_model_id": "llama3.1:8b",
    "model_type": "llm",
})

mi_config_path = "/content/mi_config_llama_stack.yaml"

with open(mi_config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

proceso_llama_stack = subprocess.Popen(
    ["llama", "stack", "run", mi_config_path, "--port", "8321"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(20)

print("Proceso vivo:", proceso_llama_stack.poll() is None)

if proceso_llama_stack.poll() is not None:
    print("--- Salida del proceso ---")
    print(proceso_llama_stack.stdout.read())
else:
    from llama_stack_client import LlamaStackClient

    client = LlamaStackClient(base_url="http://localhost:8321")
    print("Llama Stack listo.")

Proceso vivo: True
Llama Stack listo.


In [22]:
# Leer credenciales desde Colab Secrets

from google.colab import userdata

VERIFY_TOKEN = "mi_token"
WHATSAPP_TOKEN = userdata.get("WHATSAPP_ACCESS_TOKEN")
PHONE_ID = userdata.get("PHONE_NUMBER_ID")

print("Credenciales cargadas correctamente.")


Credenciales cargadas correctamente.


**2. Construye la memoria de conversación:** Crea una conversación de **Llama Stack** por número de teléfono (`client.conversations.create()`) con una memoria de respuesta.

In [23]:
# Conversacion de Llama Stack por numero de telefono

conversaciones_por_numero = {}

def obtener_conversacion(numero):
    if numero not in conversaciones_por_numero:
        conversacion = client.conversations.create()
        conversaciones_por_numero[numero] = conversacion.id
    return conversaciones_por_numero[numero]

In [24]:
# Definir la funcion de generacion de respuesta con memoria

MAX_ITEMS_HISTORIAL = 6

SYSTEM_PROMPT = (
    "Eres el asistente de una clinica veterinaria. "
    "Responde de forma breve (2-3 lineas)."
)

def recortar_historial(conversation_id):
    items = client.conversations.items.list(
        conversation_id=conversation_id,
        order="asc",
        limit=100
    )

    exceso = len(items.data) - MAX_ITEMS_HISTORIAL

    if exceso > 0:
        for item in items.data[:exceso]:
            client.conversations.items.delete(
                item.id,
                conversation_id=conversation_id
            )

def generar_respuesta(numero, texto_usuario):
    conversation_id = obtener_conversacion(numero)

    respuesta = client.responses.create(
        model="ollama/llama3.1:8b",
        input=texto_usuario,
        conversation=conversation_id,
        instructions=SYSTEM_PROMPT,
    )

    recortar_historial(conversation_id)

    return respuesta.output_text


**3. Agrega la regla fija y protege el procesamiento por número:**

   * Cuando el usuario pide hablar con una persona, se responde con un mensaje fijo de escalamiento.
   * Igual que en el Hands-On, usa un lock por número de teléfono para que dos mensajes seguidos del mismo usuario no disparen dos llamadas en paralelo sobre la misma conversación.

In [25]:
# Un lock por numero de telefono, para serializar mensajes del mismo usuario

import threading

locks_por_numero = {}  # { numero: threading.Lock() }

def obtener_lock(numero):
    if numero not in locks_por_numero:
        locks_por_numero[numero] = threading.Lock()
    return locks_por_numero[numero]


In [26]:
# Definir funcion de procesamiento: regla fija de escalamiento o respuesta de Llama

FRASES_ESCALAMIENTO = [
    "hablar con alguien",
    "hablar con una persona",
    "agente humano"
]

PALABRAS_URGENCIA = [
    "urgencia",
    "emergencia"
]

errores_procesamiento = []

def procesar_y_responder(numero, texto):
    with obtener_lock(numero):
        try:
            texto_low = texto.lower()

            if any(palabra in texto_low for palabra in PALABRAS_URGENCIA):
                enviar_respuesta(
                    numero,
                    "Si se trata de una urgencia, lleva a tu mascota a atención veterinaria inmediata."
                )

            elif any(frase in texto_low for frase in FRASES_ESCALAMIENTO):
                enviar_respuesta(
                    numero,
                    "Claro, en un momento un miembro del equipo te contacta directamente."
                )

            else:
                respuesta = generar_respuesta(numero, texto)
                enviar_respuesta(numero, respuesta)

        except Exception as e:
            errores_procesamiento.append({
                "numero": numero,
                "texto": texto,
                "error": repr(e)
            })

**4. Define los endpoints del webhook:** usa `BackgroundTasks` para responder de inmediato a Meta, delegando el procesamiento real a la función anterior.

In [28]:
# Configurar el servidor del webhook

from fastapi import FastAPI, Request, BackgroundTasks
import requests
import ollama

app = FastAPI()

mensajes_recibidos = []
usuarios_conocidos = set()

In [29]:
# Definir función de normalización de número

def normalizar_numero_mx(numero):
    if numero.startswith("521") and len(numero) == 13:
        return "52" + numero[3:]
    return numero

In [30]:
# Definir endpoint de verificacion de credenciales

@app.get("/webhook")
async def verificar_webhook(request: Request):
    params = request.query_params

    if params.get("hub.verify_token") == VERIFY_TOKEN:
        return int(params.get("hub.challenge"))

    return {"error": "token invalido"}


In [31]:
# Definir endpoint de recepcion de mensajes

@app.post("/webhook")
async def recibir_mensaje(request: Request, background_tasks: BackgroundTasks):
    body = await request.json()
    mensajes_recibidos.append(body)

    try:
        mensaje = body["entry"][0]["changes"][0]["value"]["messages"][0]
        numero = normalizar_numero_mx(mensaje["from"])
        texto = mensaje["text"]["body"]

        background_tasks.add_task(
            procesar_y_responder,
            numero,
            texto
        )

    except (KeyError, IndexError):
        pass

    return {"status": "ok"}


In [32]:
# Definir funcion de envio de respuesta

respuestas_enviadas = []

def enviar_respuesta(numero, texto):
    url = f"https://graph.facebook.com/v25.0/{PHONE_ID}/messages"

    headers = {
        "Authorization": f"Bearer {WHATSAPP_TOKEN}",
        "Content-Type": "application/json"
    }

    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {"body": texto}
    }

    r = requests.post(url, headers=headers, json=payload)

    respuestas_enviadas.append({
        "numero": numero,
        "status": r.status_code,
        "texto": r.text
    })


**5. Expón el servidor con ngrok y prueba el flujo completo:**

   * Manda una conversación de al menos 4 mensajes y confirma que el historial se recorta correctamente.
   * Manda un mensaje con una palabra de urgencia y confirma que dispara la regla fija en vez de una respuesta de Llama.

In [33]:
# Instalar e iniciar tunel publico con ngrok

!pip install pyngrok --quiet

from pyngrok import ngrok
import uvicorn
import threading

ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
tunel_publico = ngrok.connect(8000)

print("URL a configurar como Callback URL en el dashboard de Meta:")
print(tunel_publico.public_url + "/webhook")

def correr_servidor():
    uvicorn.run(app, host="0.0.0.0", port=8000)

hilo = threading.Thread(target=correr_servidor, daemon=True)
hilo.start()

URL a configurar como Callback URL en el dashboard de Meta:
https://trustee-hardware-antsy.ngrok-free.dev/webhook


INFO:     Started server process [4822]
INFO:     Waiting for application startup.


In [47]:
# Mandar el primer mensaje, mencionando el servicio

payload_1 = {
    "entry": [{"changes": [{"value": {"messages": [
        {
            "from": "526182414360",
            "text": {"body": "Quiero una cita veterinaria para mi perro"}
        }
    ]}}]}]
}

r1 = requests.post(
    "http://127.0.0.1:8000/webhook",
    json=payload_1
)

print("Mensaje 1:", r1.status_code, r1.text)


INFO:     127.0.0.1:34840 - "POST /webhook HTTP/1.1" 200 OK
Mensaje 1: 200 {"status":"ok"}


In [48]:
# Mandar un segundo mensaje que depende del primero, sin repetir el servicio

payload_2 = {
    "entry": [{"changes": [{"value": {"messages": [
        {
            "from": "526182414360",
            "text": {"body": "¿Tienen horario el sábado?"}
        }
    ]}}]}]
}

r2 = requests.post(
    "http://127.0.0.1:8000/webhook",
    json=payload_2
)

print("Mensaje 2:", r2.status_code, r2.text)

INFO:     127.0.0.1:42204 - "POST /webhook HTTP/1.1" 200 OK
Mensaje 2: 200 {"status":"ok"}


In [49]:
# Mandar un tercer mensaje de la misma conversacion

payload_3 = {
    "entry": [{"changes": [{"value": {"messages": [
        {
            "from": "526182414360",
            "text": {"body": "Es para una revisión general"}
        }
    ]}}]}]
}

r3 = requests.post(
    "http://127.0.0.1:8000/webhook",
    json=payload_3
)

print("Mensaje 3:", r3.status_code, r3.text)

INFO:     127.0.0.1:43248 - "POST /webhook HTTP/1.1" 200 OK
Mensaje 3: 200 {"status":"ok"}


In [50]:
# Mandar un cuarto mensaje de la misma conversacion

payload_4 = {
    "entry": [{"changes": [{"value": {"messages": [
        {
            "from": "526182414360",
            "text": {"body": "Prefiero una cita por la tarde"}
        }
    ]}}]}]
}

r4 = requests.post(
    "http://127.0.0.1:8000/webhook",
    json=payload_4
)

print("Mensaje 4:", r4.status_code, r4.text)

INFO:     127.0.0.1:53898 - "POST /webhook HTTP/1.1" 200 OK
Mensaje 4: 200 {"status":"ok"}


In [51]:
# Verificar respuestas, espera 1-2 minutos

respuestas_enviadas


[{'numero': '526182414360',
  'status': 200,
  'texto': '{"messaging_product":"whatsapp","contacts":[{"input":"526182414360","wa_id":"5216182414360"}],"messages":[{"id":"wamid.HBgNNTIxNjE4MjQxNDM2MBUCABEYEkUwNjRGMDZBMTMxMkREQUQyRAA="}]}'},
 {'numero': '526182414360',
  'status': 200,
  'texto': '{"messaging_product":"whatsapp","contacts":[{"input":"526182414360","wa_id":"5216182414360"}],"messages":[{"id":"wamid.HBgNNTIxNjE4MjQxNDM2MBUCABEYEjVGQzNDMDM5NDU1Mjc2OUFFQgA="}]}'},
 {'numero': '526182414360',
  'status': 200,
  'texto': '{"messaging_product":"whatsapp","contacts":[{"input":"526182414360","wa_id":"5216182414360"}],"messages":[{"id":"wamid.HBgNNTIxNjE4MjQxNDM2MBUCABEYEjhEMTA1NUFEMTAxM0UwNjgxNQA="}]}'},
 {'numero': '526182414360',
  'status': 200,
  'texto': '{"messaging_product":"whatsapp","contacts":[{"input":"526182414360","wa_id":"5216182414360"}],"messages":[{"id":"wamid.HBgNNTIxNjE4MjQxNDM2MBUCABEYEjI5OURDMzA3RDNERTY4OEYzQgA="}]}'}]

In [39]:
# Ver mensajes recibidos

mensajes_recibidos


[{'entry': [{'changes': [{'value': {'messages': [{'from': '526182414360',
         'text': {'body': 'Quiero una cita veterinaria para mi perro'}}]}}]}]},
 {'entry': [{'changes': [{'value': {'messages': [{'from': '526182414360',
         'text': {'body': '¿Tienen horario el sábado?'}}]}}]}]},
 {'entry': [{'changes': [{'value': {'messages': [{'from': '526182414360',
         'text': {'body': 'Es para una revisión general'}}]}}]}]},
 {'entry': [{'changes': [{'value': {'messages': [{'from': '526182414360',
         'text': {'body': 'Prefiero una cita por la tarde'}}]}}]}]}]

In [52]:
# Verificar almacenamiento en Llama Stack

numero = normalizar_numero_mx("526182414360")
conv_id = conversaciones_por_numero[numero]

items = client.conversations.items.list(
    conversation_id=conv_id,
    order="asc",
    limit=100
)

print("Elementos conservados:", len(items.data))

for item in items.data:
    print(item.role, "->", item.content)

Elementos conservados: 6
user -> [OpenAIResponseMessageOutputContentListOpenAIResponseInputMessageContentTextOpenAIResponseInputMessageContentImageOpenAIResponseInputMessageContentFileOpenAIResponseInputMessageContentText(text='¿Tienen horario el sábado?', type='input_text')]
assistant -> [OpenAIResponseMessageOutputContentListOpenAIResponseOutputMessageContentOutputTextOutputOpenAIResponseContentPartRefusalOpenAIResponseOutputMessageContentOutputTextOutput(text='Sí, tenemos horario el sábado. Podemos ofrecerte un horario al mediodía o a la tarde. ¿Cuál prefieres?', annotations=[], logprobs=[], type='output_text')]
user -> [OpenAIResponseMessageOutputContentListOpenAIResponseInputMessageContentTextOpenAIResponseInputMessageContentImageOpenAIResponseInputMessageContentFileOpenAIResponseInputMessageContentText(text='Es para una revisión general', type='input_text')]
assistant -> [OpenAIResponseMessageOutputContentListOpenAIResponseOutputMessageContentOutputTextOutputOpenAIResponseContent

In [53]:
# Probar regla fija de urgencia

payload_urgencia = {
    "entry": [{"changes": [{"value": {"messages": [
        {
            "from": "526182414360",
            "text": {"body": "Es una urgencia, mi perro necesita atención"}
        }
    ]}}]}]
}

r_urgencia = requests.post(
    "http://127.0.0.1:8000/webhook",
    json=payload_urgencia
)

print("Mensaje de urgencia:", r_urgencia.status_code, r_urgencia.text)

INFO:     127.0.0.1:46010 - "POST /webhook HTTP/1.1" 200 OK
Mensaje de urgencia: 200 {"status":"ok"}


In [54]:
# Verificar respuesta de urgencia

respuestas_enviadas[-1]

{'numero': '526182414360',
 'status': 200,
 'texto': '{"messaging_product":"whatsapp","contacts":[{"input":"526182414360","wa_id":"5216182414360"}],"messages":[{"id":"wamid.HBgNNTIxNjE4MjQxNDM2MBUCABEYEjlDMTRBRkRFQTU5QjlCQ0JCRQA="}]}'}

In [55]:
# Confirmar que la urgencia no fue procesada por Llama

items_urgencia = client.conversations.items.list(
    conversation_id=conv_id,
    order="asc",
    limit=100
)

print("Elementos en Llama Stack después de la urgencia:", len(items_urgencia.data))

for item in items_urgencia.data:
    print(item.role, "->", item.content)

Elementos en Llama Stack después de la urgencia: 6
user -> [OpenAIResponseMessageOutputContentListOpenAIResponseInputMessageContentTextOpenAIResponseInputMessageContentImageOpenAIResponseInputMessageContentFileOpenAIResponseInputMessageContentText(text='¿Tienen horario el sábado?', type='input_text')]
assistant -> [OpenAIResponseMessageOutputContentListOpenAIResponseOutputMessageContentOutputTextOutputOpenAIResponseContentPartRefusalOpenAIResponseOutputMessageContentOutputTextOutput(text='Sí, tenemos horario el sábado. Podemos ofrecerte un horario al mediodía o a la tarde. ¿Cuál prefieres?', annotations=[], logprobs=[], type='output_text')]
user -> [OpenAIResponseMessageOutputContentListOpenAIResponseInputMessageContentTextOpenAIResponseInputMessageContentImageOpenAIResponseInputMessageContentFileOpenAIResponseInputMessageContentText(text='Es para una revisión general', type='input_text')]
assistant -> [OpenAIResponseMessageOutputContentListOpenAIResponseOutputMessageContentOutputTextO

In [56]:
# Probar regla fija de escalamiento

payload_escalamiento = {
    "entry": [{"changes": [{"value": {"messages": [
        {
            "from": "526182414360",
            "text": {"body": "Quiero hablar con una persona"}
        }
    ]}}]}]
}

r_escalamiento = requests.post(
    "http://127.0.0.1:8000/webhook",
    json=payload_escalamiento
)

print("Mensaje de escalamiento:", r_escalamiento.status_code, r_escalamiento.text)

INFO:     127.0.0.1:42886 - "POST /webhook HTTP/1.1" 200 OK
Mensaje de escalamiento: 200 {"status":"ok"}


In [57]:
# Confirmar que el escalamiento no fue procesado por Llama

items_escalamiento = client.conversations.items.list(
    conversation_id=conv_id,
    order="asc",
    limit=100
)

print("Elementos en Llama Stack después del escalamiento:", len(items_escalamiento.data))

for item in items_escalamiento.data:
    print(item.role, "->", item.content)

Elementos en Llama Stack después del escalamiento: 6
user -> [OpenAIResponseMessageOutputContentListOpenAIResponseInputMessageContentTextOpenAIResponseInputMessageContentImageOpenAIResponseInputMessageContentFileOpenAIResponseInputMessageContentText(text='¿Tienen horario el sábado?', type='input_text')]
assistant -> [OpenAIResponseMessageOutputContentListOpenAIResponseOutputMessageContentOutputTextOutputOpenAIResponseContentPartRefusalOpenAIResponseOutputMessageContentOutputTextOutput(text='Sí, tenemos horario el sábado. Podemos ofrecerte un horario al mediodía o a la tarde. ¿Cuál prefieres?', annotations=[], logprobs=[], type='output_text')]
user -> [OpenAIResponseMessageOutputContentListOpenAIResponseInputMessageContentTextOpenAIResponseInputMessageContentImageOpenAIResponseInputMessageContentFileOpenAIResponseInputMessageContentText(text='Es para una revisión general', type='input_text')]
assistant -> [OpenAIResponseMessageOutputContentListOpenAIResponseOutputMessageContentOutputTex

In [58]:
# Auditoría final de requisitos del challenge

items_final = client.conversations.items.list(
    conversation_id=conv_id,
    order="asc",
    limit=100
)

historial_texto = str([item.content for item in items_final.data])
eventos_texto = str(mensajes_recibidos)

mensajes_conversacion = [
    "Quiero una cita veterinaria para mi perro",
    "¿Tienen horario el sábado?",
    "Es para una revisión general",
    "Prefiero una cita por la tarde"
]

pruebas = {
    "Conversación de al menos 4 mensajes":
        all(mensaje in eventos_texto for mensaje in mensajes_conversacion),

    "Historial limitado a 6 elementos":
        len(items_final.data) == MAX_ITEMS_HISTORIAL,

    "Mensaje más antiguo eliminado":
        "Quiero una cita veterinaria para mi perro" not in historial_texto,

    "Urgencia recibida por el webhook":
        "Es una urgencia, mi perro necesita atención" in eventos_texto,

    "Urgencia interceptada antes de Llama":
        "Es una urgencia, mi perro necesita atención" not in historial_texto,

    "Escalamiento recibido por el webhook":
        "Quiero hablar con una persona" in eventos_texto,

    "Escalamiento interceptado antes de Llama":
        "Quiero hablar con una persona" not in historial_texto,

    "Envíos a WhatsApp sin error":
        len(respuestas_enviadas) >= 6
        and all(r["status"] == 200 for r in respuestas_enviadas),

    "Sin errores internos de procesamiento":
        len(errores_procesamiento) == 0,

    "Lock persistente para el mismo número":
        obtener_lock(numero) is obtener_lock(numero)
}

print("AUDITORÍA DEL CHALLENGE\n")

for nombre, resultado in pruebas.items():
    print(("PASS" if resultado else "FAIL"), "-", nombre)

assert all(pruebas.values()), "La auditoría detectó al menos un requisito incumplido."

print("\nAuditoría final: PASS")

AUDITORÍA DEL CHALLENGE

PASS - Conversación de al menos 4 mensajes
PASS - Historial limitado a 6 elementos
PASS - Mensaje más antiguo eliminado
PASS - Urgencia recibida por el webhook
PASS - Urgencia interceptada antes de Llama
PASS - Escalamiento recibido por el webhook
PASS - Escalamiento interceptado antes de Llama
PASS - Envíos a WhatsApp sin error
PASS - Sin errores internos de procesamiento
PASS - Lock persistente para el mismo número

Auditoría final: PASS


### Observación crítica

Durante las pruebas, el modelo generó horarios y disponibilidad de citas que no fueron proporcionados como datos de entrada ni consultados en una fuente externa. Aunque el flujo conversacional funciona correctamente, estas respuestas muestran que el modelo puede generar información operativa no verificada. En una aplicación real, datos como horarios, disponibilidad, personal y citas deberían obtenerse de una fuente confiable del negocio, mientras que el modelo de lenguaje debería utilizarse principalmente para interpretar la solicitud y redactar la respuesta.

In [59]:
# Finalizar proceso de tunelización

ngrok.kill()
print("Proceso de ngrok cerrado.")


Proceso de ngrok cerrado.
